# RoadWatch BD: how much does BadODD's split leak? (Kaggle GPU)

The conference-paper experiment. BadODD's official split is by image: every video has frames in train *and*
test, a typical test frame only ~59 frames from a training frame. **Question:** how much higher does a
fine-tuned detector score on a test image when it has trained on other frames of the same video?

**Design (paired, `scripts/make_group_split.py paired`).** The 12 test videos of
`configs/splits/badodd_video_v1.json` give one shared test set (every other frame, 1,019 images). Two training
sets of equal size (5,967 images):
- **clean**: never sees the test videos.
- **leaky**: the same, but with ~1,000 random images swapped for the *other* frames of the test videos.

The test images, training size, validation set and recipe are identical, so the score difference is the leak.

| `RUNS` entry | What it does | T4 time (rough) |
|---|---|---|
| `zeroshot` | COCO RT-DETR r18 and COCO YOLO11s on the shared and the official test set | ~10 min |
| `rtdetr_leaky`, `rtdetr_clean` | fine-tune RT-DETR r18 (Apache-2.0), 30 epochs, evaluate | 1–2 h each |
| `yolo_leaky`, `yolo_clean` | fine-tune YOLO11s (Ultralytics, AGPL-3.0; fine for research), 100 epochs, evaluate | 2–3 h each |
| `rtdetr_official`, `yolo_official` | optional: the same on BadODD's own split, to compare with published numbers | as above |

A Kaggle session stops after 12 hours, so run it in two sessions: first `zeroshot` and both RT-DETR runs, then
both YOLO runs (edit `RUNS` in section 2).

**Before running:** Accelerator → **GPU T4**, **Internet on** (phone-verified account). Use
**Save Version → Save & Run All (Commit)** so it keeps running with the browser closed. Download
`bench_results.zip` from the **Output** tab afterwards.

Credit: BadODD, Baig et al., arXiv:2401.10659, Zenodo record 13823687, CC BY 4.0.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!rm -rf /kaggle/temp/roadwatch-bd && git clone --depth 1 https://github.com/muttakin210/roadwatch-bd.git /kaggle/temp/roadwatch-bd
!pip install -q "transformers>=4.50" timm "torchmetrics>=1.4" "pycocotools>=2.0.8" opencv-python-headless pyyaml "ultralytics>=8.3"

## 1. Download BadODD, convert it, build the paired split
`badodd_unified` keeps the official split; `badodd_paired/{leaky,clean}` are hard links to the same images.

In [ ]:
%%bash
set -e
mkdir -p /kaggle/temp/badodd && cd /kaggle/temp/badodd
if [ ! -d extracted ]; then
  wget -q -O badodd.zip "https://zenodo.org/api/records/13823687/files/badodd.zip/content"
  echo "6a0d3983b379302ab79a99baf7d47d74  badodd.zip" | md5sum -c -
  unzip -q -o badodd.zip -d extracted && rm badodd.zip
fi
cd /kaggle/temp/roadwatch-bd/ml
[ -d /kaggle/temp/badodd_unified ] || python scripts/convert_datasets.py --dataset badodd \
    --src /kaggle/temp/badodd/extracted --dst /kaggle/temp/badodd_unified
[ -d /kaggle/temp/badodd_paired ] || python scripts/make_group_split.py paired --src /kaggle/temp/badodd_unified \
    --manifest configs/splits/badodd_video_v1.json --dst /kaggle/temp/badodd_paired
for d in badodd_unified badodd_paired/leaky badodd_paired/clean; do
  echo "$d: $(ls /kaggle/temp/$d/train/images | wc -l) train / $(ls /kaggle/temp/$d/val/images | wc -l) val / $(ls /kaggle/temp/$d/test/images | wc -l) test"
done

## 2. Choose what to run
Every score is AP@50 from `eval/eval_detector.py`, with 95 % confidence intervals that resample whole
videos (frames of one video are not independent).

In [ ]:
RUNS = ["zeroshot", "rtdetr_leaky", "rtdetr_clean", "yolo_leaky", "yolo_clean"]  # trim per session; optional: "rtdetr_official", "yolo_official"
EPOCHS_RTDETR, EPOCHS_YOLO, IMGSZ, SEED = 30, 100, 640, 0

import json, os, subprocess, sys
from pathlib import Path

ML = Path("/kaggle/temp/roadwatch-bd/ml")
os.chdir(ML)
sys.path.insert(0, str(ML))
from roadwatch.types import CLASSES

PAIRED = Path("/kaggle/temp/badodd_paired")
DATA = {"leaky": PAIRED / "leaky", "clean": PAIRED / "clean", "official": Path("/kaggle/temp/badodd_unified")}
TEST = {"paired": PAIRED / "clean" / "test", "official": DATA["official"] / "test"}  # leaky/test = same images
OUT = Path("/kaggle/working/bench")
RES = OUT / "results"
RES.mkdir(parents=True, exist_ok=True)


def sh(cmd):
    # run a command, streaming its output here; stop the notebook if it fails
    cmd = [str(c) for c in cmd]
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"failed ({p.returncode}): {' '.join(cmd)}")


def evaluate(name, test, *extra):
    sh([sys.executable, "eval/eval_detector.py", "--split", TEST[test], "--out", RES / f"{name}_on_{test}",
        "--bootstrap-unit", "video", *extra])


def test_of(train):
    return "official" if train == "official" else "paired"

## 3. Zero-shot COCO detectors
Never trained on BadODD: a reference point for how much fine-tuning adds.

In [ ]:
if "zeroshot" in RUNS:
    for test in TEST:
        evaluate("coco_rtdetr_r18", test)
        evaluate("coco_yolo11s", test, "--detector", "ultralytics", "--model-id", "yolo11s.pt", "--imgsz", IMGSZ)

## 4. Fine-tune RT-DETR r18
Same recipe for every training set (`scripts/train_detector.py`); the best epoch by validation mAP@50 is kept.

In [ ]:
for train in DATA:
    if f"rtdetr_{train}" not in RUNS:
        continue
    out = OUT / f"rtdetr_r18_{train}"
    sh([sys.executable, "scripts/train_detector.py", "--data", DATA[train], "--out", out,
        "--epochs", EPOCHS_RTDETR, "--batch", 8, "--workers", 4, "--max-val", 500, "--seed", SEED])
    evaluate(f"bd_rtdetr_r18_{train}", test_of(train), "--detector", "hf", "--model-id", out / "best")

## 5. Fine-tune YOLO11s
Ultralytics' standard recipe (mosaic and other augmentations), same settings for every training set.

In [ ]:
from ultralytics import YOLO

for train in DATA:
    if f"yolo_{train}" not in RUNS:
        continue
    cfg = OUT / f"badodd_{train}.yaml"
    cfg.write_text(json.dumps({"path": str(DATA[train]), "train": "train/images", "val": "val/images",
                               "names": CLASSES}))
    YOLO("yolo11s.pt").train(data=str(cfg), epochs=EPOCHS_YOLO, imgsz=IMGSZ, batch=16, workers=4, seed=SEED,
                             project=str(OUT), name=f"yolo11s_{train}", exist_ok=True, plots=False)
    evaluate(f"bd_yolo11s_{train}", test_of(train), "--detector", "ultralytics",
             "--model-id", OUT / f"yolo11s_{train}" / "weights" / "best.pt", "--imgsz", IMGSZ)

## 6. Results: the leakage effect
`eval/compare_runs.py` rescores leaky and clean with the *same* video-bootstrap draws, so its interval is for
the difference itself, and it counts in how many of the 12 test videos the leaky model wins.

In [ ]:
print(f"{'result':36s} {'images':>6s} {'mAP50':>6s}  {'95% CI (videos)':15s}  mAP50, classes the model can output")
for p in sorted(RES.glob("*.json")):
    r = json.loads(p.read_text())
    lo, hi = r["map50_all_classes_ci95"]
    print(f"{p.stem:36s} {r['n_images']:6d} {r['map50_all_classes']:6.3f}  {lo:.3f}–{hi:.3f}      "
          f"{r['map50_predictable_classes']:.3f}")

for det in ("rtdetr_r18", "yolo11s"):
    a, b = RES / f"bd_{det}_leaky_on_paired", RES / f"bd_{det}_clean_on_paired"
    if a.with_suffix(".json").exists() and b.with_suffix(".json").exists():
        print()
        sh([sys.executable, "eval/compare_runs.py", "--split", TEST["paired"], "--a", a, "--b", b,
            "--out", OUT / "leakage" / det])

In [ ]:
# results, training logs, best weights and the raw detections (.pkl, for re-scoring on your PC without a GPU)
!cd /kaggle/working && rm -f bench_results.zip && zip -qr bench_results.zip bench -x "bench/*/last/*" "bench/*/weights/last.pt" && ls -lh bench_results.zip

Unzip `bench_results.zip` into `D:\Traffic AI model project\ml\eval\kaggle\` on your PC. `results/*.md` and
`leakage/*.md` are readable as they are; the paper's tables are built from the `.json` files.